In [13]:
# Preamble
from PIL import Image
import os
import shutil
from tqdm import tqdm
os.chdir("/Users/brenden/Desktop/motorVAE")

# Parameters of preprocessing

In [14]:
suffix_low = 1
suffix_high = 4
target_resolution = 256

# Only first few images in group

In [15]:
def copy_first_images(input_folder, output_folder, suffix_low, suffix_high):
    """
    Copies all images with suffixes from suffix_low through suffix_high from the input folder to the output folder.

    Parameters:
    - input_folder (str): Path to the folder containing images.
    - output_folder (str): Path to the folder where copied images will be saved.
    - suffix_low (int): The lowest suffix number to include.
    - suffix_high (int): The highest suffix number to include.
    """

    # Input validation for suffix parameters
    if not isinstance(suffix_low, int) or not isinstance(suffix_high, int):
        raise TypeError("Both suffix_low and suffix_high must be integers")
    if suffix_high < suffix_low:
        raise ValueError("suffix_high must be greater than or equal to suffix_low")

    os.makedirs(output_folder, exist_ok=True)

    # Dynamically create the list of valid suffixes based on the range
    valid_suffixes = tuple(f"_{i}.png" for i in range(suffix_low, suffix_high + 1))

    # Get all files from the input folder that end with any of the valid suffixes
    filenames = [f for f in os.listdir(input_folder) if any(f.endswith(suffix) for suffix in valid_suffixes)]

    copied_count = 0
    for filename in filenames:
        src_path = os.path.join(input_folder, filename)
        dest_path = os.path.join(output_folder, filename)

        if os.path.exists(src_path):
            shutil.copy2(src_path, dest_path)
            copied_count += 1
        else:
            print(f"⚠️ Warning: File does not exist and was skipped: {src_path}")

    print(f"\n✅ Done! Copied {copied_count} images from {input_folder} to {output_folder}")

# Example usage
input_folders = [
    "data/raw_vehicle_images/vehicle_images2007-2011", 
    "data/raw_vehicle_images/vehicle_images2012-2015",
    "data/raw_vehicle_images/vehicle_images2016-2018",
    "data/raw_vehicle_images/vehicle_images2019-2025"
]

output_folder = f"data/raw_vehicle_images/evox_640x480_{suffix_low}-{suffix_high}"

os.makedirs(output_folder, exist_ok=True)
for folder in input_folders:
    copy_first_images(folder, output_folder, suffix_low, suffix_high)



✅ Done! Copied 10555 images from data/raw_vehicle_images/vehicle_images2007-2011 to data/raw_vehicle_images/evox_640x480_1-4

✅ Done! Copied 9308 images from data/raw_vehicle_images/vehicle_images2012-2015 to data/raw_vehicle_images/evox_640x480_1-4

✅ Done! Copied 7734 images from data/raw_vehicle_images/vehicle_images2016-2018 to data/raw_vehicle_images/evox_640x480_1-4

✅ Done! Copied 18837 images from data/raw_vehicle_images/vehicle_images2019-2025 to data/raw_vehicle_images/evox_640x480_1-4


# Preprocessing function

In [25]:
def preprocess(image_folder, filename, output_folder, target_resolution):
    """
    Replaces the transparent background of a PNG image with white, converts it to grayscale,
    and resizes it to target dimensions while preserving aspect ratio with white padding.

    Args:
        image_folder (str): The folder containing the input PNG image.
        filename (str): The filename of the image to process.
        output_path (str): The path to save the output image.
    """
    try:
        img = Image.open(os.path.join(image_folder, filename)).convert("RGBA")
        
        # Create a white background image with an alpha channel
        white_bg = Image.new("RGBA", img.size, (255, 255, 255, 255))

        # Composite the image with the white background to blend edges properly
        blended = Image.alpha_composite(white_bg, img)

        # Convert to RGB to remove alpha channel
        rgb_img = blended.convert("RGB")

        # Convert to greyscale
        grayscale_img = rgb_img.convert("L")  

        # Define cropping box (left, upper, right, lower)
        width, height = grayscale_img.size
        crop_box = (5, 20, width - 25, height - 20)
        cropped_img = grayscale_img.crop(crop_box)
        
        # Target dimensions
        target_width, target_height = target_resolution, target_resolution
        
        # Calculate the aspect ratio
        img_width, img_height = cropped_img.size
        aspect_ratio = img_width / img_height
        
        # Determine new dimensions that preserve aspect ratio
        if aspect_ratio > 1:  # Width is greater than height
            new_width = target_width
            new_height = int(target_width / aspect_ratio)
        else:  # Height is greater than or equal to width
            new_height = target_height
            new_width = int(target_height * aspect_ratio)
        
        # Resize the image while preserving aspect ratio
        resized_img = cropped_img.resize((new_width, new_height), Image.LANCZOS)
        
        # Create a new white image with target dimensions
        final_img = Image.new("L", (target_width, target_height), 255)
        
        # Calculate position to paste resized image (centered)
        paste_x = (target_width - new_width) // 2
        paste_y = (target_height - new_height) // 2
        
        # Paste the resized image onto the white background
        final_img.paste(resized_img, (paste_x, paste_y))

        # Save as PNG
        final_img.save(os.path.join(output_folder, filename), "PNG")

    except Exception as e:
        print(f"Error processing {os.path.join(image_folder, filename)}: {e}")

# Preprocess wrapper

In [27]:
def preprocess_wrapper(image_folder, output_path):
    """
    Processes all PNG images in a folder, replacing transparent backgrounds with white and converting to RGB.

    Args:
        folder_path (str): The path to the folder containing PNG images.
    """
    image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(".png")]
    for filename in tqdm(image_files, desc="Processing images", unit="img"):
        preprocess(image_folder, filename, output_path, target_resolution)  # Overwrite with the same name

# Input-Output
input_folder = f"data/raw_vehicle_images/evox_640x480_{suffix_low}-{suffix_high}"
output_folder = f"data/evox_{target_resolution}x{target_resolution}_{suffix_low}-{suffix_high}"
os.makedirs(output_folder, exist_ok=True)

# Run
preprocess_wrapper(input_folder, output_folder)

Processing images: 100%|██████████| 46434/46434 [11:17<00:00, 68.54img/s]
